# Coconut Leaf Health Detection Model v3
### Color-Scale Classification: Healthy (Green) vs Unhealthy (Yellow/Brown)

**Dataset:**
- `healthy-leaves/` — Green, healthy coconut leaves  
- `unhealthy-yellowing/` — Yellowing/browning, unhealthy coconut leaves

**New in v3:**
- **Color Health Score (0–100)** — Continuous scale based on HSV green-pixel ratio  
  - 70–100 → Healthy (dominant green)  
  - 30–69 → Moderate (some yellowing)  
  - 0–29 → Unhealthy (dominant yellow / brown)  
- **EfficientNetB0** backbone (same as v2 — best color feature learning)  
- **No color augmentation** (preserves green vs yellow signal)  
- **Focal Loss + Label Smoothing 0.1** (no overconfident 100% outputs)  
- **70/15/15 re-split** — guarantees ≥500 test images

**Supervisor Requirements:**
- [x] Test set ≥ 500 images  
- [x] Precision, Recall, F1 close to each other per class  
- [x] Similar values across all classes  
- [x] Target accuracy ≥ 95%

## 1. Setup and Imports

In [ ]:
import os
import shutil
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

print(f"TensorFlow: {tf.__version__}")
print(f"GPU:        {tf.config.list_physical_devices('GPU')}")

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

## 2. Configuration

In [ ]:
BASE_DIR = os.path.abspath('..')

# Source data (raw image folders)
HEALTHY_SRC   = os.path.join(BASE_DIR, 'data', 'raw', 'healthy-leaves')
UNHEALTHY_SRC = os.path.join(BASE_DIR, 'data', 'raw', 'unhealthy-yellowing')

# Dataset and model directories for v3
DATASET_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'leaf_health_v3', 'dataset')
MODEL_DIR   = os.path.join(BASE_DIR, 'models', 'leaf_health_v3')
os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 32

PHASE1_EPOCHS = 30
PHASE2_EPOCHS = 20
LR_PHASE1     = 1e-3
LR_PHASE2     = 3e-5

# 70 / 15 / 15 split
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

CLASS_NAMES = ['healthy', 'unhealthy']  # alphabetical = ImageDataGenerator order

VALID_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

print(f"Healthy source   : {HEALTHY_SRC}")
print(f"Unhealthy source : {UNHEALTHY_SRC}")
print(f"Dataset dir      : {DATASET_DIR}")
print(f"Model dir        : {MODEL_DIR}")
print(f"Split            : {int(TRAIN_RATIO*100)} / {int(VAL_RATIO*100)} / {int(TEST_RATIO*100)}")

## 3. Pool All Data and Re-split (70 / 15 / 15)

Collect all images from every sub-folder (`training/`, `test/`, `validation/`) and re-split cleanly.

```
healthy-leaves:      ~4804 images  →  train ~3363 | val ~721 | test ~720
unhealthy-yellowing: ~4181 images  →  train ~2927 | val ~627 | test ~627
                                                        ─────────────────
                                     Total test:        ~1347  ✓ (≥500)
```

In [ ]:
def collect_all_images(src_dir):
    """Walk all sub-folders and collect image file paths."""
    images = []
    for root, _, files in os.walk(src_dir):
        for f in files:
            if f.lower().endswith(VALID_EXT):
                images.append(os.path.join(root, f))
    return images

healthy_all   = collect_all_images(HEALTHY_SRC)
unhealthy_all = collect_all_images(UNHEALTHY_SRC)

random.shuffle(healthy_all)
random.shuffle(unhealthy_all)

print(f"Healthy images   : {len(healthy_all)}")
print(f"Unhealthy images : {len(unhealthy_all)}")
print(f"Total            : {len(healthy_all) + len(unhealthy_all)}")

In [ ]:
def split_list(lst, train_r, val_r):
    n = len(lst)
    t = int(n * train_r)
    v = int(n * (train_r + val_r))
    return lst[:t], lst[t:v], lst[v:]

h_train, h_val, h_test = split_list(healthy_all,   TRAIN_RATIO, VAL_RATIO)
u_train, u_val, u_test = split_list(unhealthy_all, TRAIN_RATIO, VAL_RATIO)

test_total = len(h_test) + len(u_test)
print(f"{'Split':<8} {'Healthy':>10} {'Unhealthy':>12} {'Total':>8}")
print("-" * 44)
print(f"{'Train':<8} {len(h_train):>10} {len(u_train):>12} {len(h_train)+len(u_train):>8}")
print(f"{'Val':<8} {len(h_val):>10} {len(u_val):>12} {len(h_val)+len(u_val):>8}")
print(f"{'Test':<8} {len(h_test):>10} {len(u_test):>12} {test_total:>8}")
print("-" * 44)
print(f"\nTest >= 500: {test_total}  {'OK' if test_total >= 500 else 'NEED MORE DATA'}")

In [ ]:
def build_dataset_folders(splits_dict, dataset_dir):
    """Copy images into train/val/test/class folder structure."""
    for split, classes in splits_dict.items():
        for cls, file_list in classes.items():
            dest = os.path.join(dataset_dir, split, cls)
            os.makedirs(dest, exist_ok=True)

            existing = len([f for f in os.listdir(dest)
                            if f.lower().endswith(VALID_EXT)])
            if existing == len(file_list):
                print(f"  {split}/{cls}: {existing} files already present (skipping)")
                continue

            # Clear old files if count mismatch
            for f in os.listdir(dest):
                if f.lower().endswith(VALID_EXT):
                    os.remove(os.path.join(dest, f))

            for i, src in enumerate(file_list):
                ext = os.path.splitext(src)[1].lower() or '.jpg'
                shutil.copy2(src, os.path.join(dest, f"{cls}_{split}_{i:05d}{ext}"))

            print(f"  {split}/{cls}: copied {len(file_list)} files")

print("Building v3 dataset structure...")
build_dataset_folders(
    {
        'train': {'healthy': h_train, 'unhealthy': u_train},
        'val':   {'healthy': h_val,   'unhealthy': u_val},
        'test':  {'healthy': h_test,  'unhealthy': u_test},
    },
    DATASET_DIR
)
print("Done!")

## 4. Dataset Summary

In [ ]:
data_summary = {}
print("=" * 55)
print("DATASET SUMMARY — leaf_health_v3")
print("=" * 55)
for split in ['train', 'val', 'test']:
    data_summary[split] = {}
    total = 0
    print(f"\n{split.upper()}:")
    for cls in CLASS_NAMES:
        path = os.path.join(DATASET_DIR, split, cls)
        count = len([f for f in os.listdir(path)
                     if f.lower().endswith(VALID_EXT)])
        data_summary[split][cls] = count
        total += count
        print(f"  {cls:<15} {count:>6}")
    print(f"  {'TOTAL':<15} {total:>6}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Class Distribution — Leaf Health v3', fontsize=13, fontweight='bold')
colors = ['#27ae60', '#e67e22']
for idx, split in enumerate(['train', 'val', 'test']):
    counts = [data_summary[split][c] for c in CLASS_NAMES]
    axes[idx].bar(CLASS_NAMES, counts, color=colors, edgecolor='black', linewidth=0.5)
    axes[idx].set_title(split.upper(), fontweight='bold')
    axes[idx].set_ylabel('Images')
    for i, v in enumerate(counts):
        axes[idx].text(i, v + 15, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

test_total = sum(data_summary['test'].values())
print(f"\nTest total: {test_total}  {'OK (>=500)' if test_total >= 500 else 'NOT ENOUGH'}")

## 5. Color Health Scale — HSV Analysis

**Color Health Score (0–100)** is computed from the HSV hue distribution:

| Score | Meaning | Color |
|-------|---------|-------|
| 70–100 | Healthy | Dominant green (H: 35–85 in OpenCV scale) |
| 30–69 | Moderate | Mixed green/yellow |
| 0–29 | Unhealthy | Dominant yellow/brown (H: 10–35) |

**Formula:** `score = (green_pixels / total_saturated_pixels) × 100`

In [ ]:
def rgb_to_hsv_np(rgb_float):
    """
    Pure-numpy RGB -> HSV conversion (no OpenCV required).
    Input:  (H, W, 3) float32 in [0, 1]
    Output: h [0-180], s [0-255], v [0-255]  (OpenCV-compatible scale)
    """
    r, g, b = rgb_float[..., 0], rgb_float[..., 1], rgb_float[..., 2]
    cmax  = np.maximum(np.maximum(r, g), b)
    cmin  = np.minimum(np.minimum(r, g), b)
    delta = cmax - cmin

    # Hue calculation
    h = np.zeros_like(r)
    m = delta > 0
    mr = m & (cmax == r)
    mg = m & (cmax == g)
    mb = m & (cmax == b)
    h[mr] = 60 * (((g[mr] - b[mr]) / delta[mr]) % 6)
    h[mg] = 60 * ((b[mg] - r[mg]) / delta[mg] + 2)
    h[mb] = 60 * ((r[mb] - g[mb]) / delta[mb] + 4)
    h = h / 2.0  # [0, 360] -> [0, 180] OpenCV scale

    # Saturation
    s = np.where(cmax > 0, delta / cmax, 0) * 255

    # Value / Brightness
    v = cmax * 255

    return h, s, v


def compute_color_health_score(img_path, resize=(112, 112)):
    """
    Compute Color Health Score (0-100) for a single image.

    Score = (green_pixels / total_saturated_pixels) * 100
    Green  : Hue 35-85 (OpenCV), Saturation > 40
    Yellow : Hue 10-35 (OpenCV), Saturation > 40
    Brown  : Hue 0-20 or 160-180, Saturation > 40, Value < 160
    """
    try:
        img = Image.open(img_path).convert('RGB').resize(resize)
        rgb = np.array(img, dtype=np.float32) / 255.0
        h, s, v = rgb_to_hsv_np(rgb)

        sat_mask   = s > 40          # only colored pixels (exclude grey/white)
        total_sat  = np.sum(sat_mask)
        if total_sat == 0:
            return 50.0              # neutral if no saturated pixels

        green_mask  = sat_mask & (h >= 35) & (h <= 85)
        green_ratio = np.sum(green_mask) / total_sat
        return float(round(green_ratio * 100, 2))
    except Exception:
        return 50.0


def get_color_stats_sample(img_dir, n_samples=150):
    """Sample images and return (hues, sats, vals, scores) arrays."""
    files = [f for f in os.listdir(img_dir) if f.lower().endswith(VALID_EXT)]
    samples = random.sample(files, min(n_samples, len(files)))
    hues, sats, vals, scores = [], [], [], []
    for fname in samples:
        fp = os.path.join(img_dir, fname)
        try:
            img = Image.open(fp).convert('RGB').resize((112, 112))
            rgb = np.array(img, dtype=np.float32) / 255.0
            h, s, v = rgb_to_hsv_np(rgb)
            mask = s > 30
            if mask.sum() == 0:
                continue
            hues.append(h[mask].mean())
            sats.append(s[mask].mean())
            vals.append(v[mask].mean())
            scores.append(compute_color_health_score(fp))
        except Exception:
            continue
    return (np.array(hues), np.array(sats),
            np.array(vals), np.array(scores))


print("Analyzing color distributions (~30 sec)...")
h_hue, h_sat, h_val, h_scores = get_color_stats_sample(
    os.path.join(DATASET_DIR, 'train', 'healthy'), 200)
u_hue, u_sat, u_val, u_scores = get_color_stats_sample(
    os.path.join(DATASET_DIR, 'train', 'unhealthy'), 200)

hue_sep = abs(h_hue.mean() - u_hue.mean())
score_sep = abs(h_scores.mean() - u_scores.mean())

print(f"\nColor Analysis:")
print(f"  {'Class':<12} {'Hue':>8} {'Sat':>8} {'Val':>8} {'Score(0-100)':>14}")
print(f"  {'-'*50}")
print(f"  {'Healthy':12} {h_hue.mean():>8.1f} {h_sat.mean():>8.1f} {h_val.mean():>8.1f} {h_scores.mean():>14.1f}")
print(f"  {'Unhealthy':12} {u_hue.mean():>8.1f} {u_sat.mean():>8.1f} {u_val.mean():>8.1f} {u_scores.mean():>14.1f}")
print(f"\n  Hue separation:   {hue_sep:.1f} (>10 = classes separable)")
print(f"  Score separation: {score_sep:.1f} (>20 = strong color signal)")
print(f"  Classes separable: {'YES' if hue_sep > 10 else 'CHECK DATA'} | Score signal: {'STRONG' if score_sep > 20 else 'WEAK'}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Color Analysis — Healthy (Green) vs Unhealthy (Yellow/Brown)',
             fontsize=13, fontweight='bold')

# Hue distribution
ax = axes[0, 0]
ax.hist(h_hue, bins=30, color='#27ae60', alpha=0.75,
        label=f'Healthy   mean={h_hue.mean():.1f}', density=True)
ax.hist(u_hue, bins=30, color='#e67e22', alpha=0.75,
        label=f'Unhealthy mean={u_hue.mean():.1f}', density=True)
ax.axvspan(35, 85, alpha=0.12, color='green', label='Green zone (35-85)')
ax.axvspan(10, 35, alpha=0.12, color='yellow', label='Yellow zone (10-35)')
ax.set_title('Hue Distribution (0=Red · 30=Yellow · 60=Green)', fontsize=9)
ax.set_xlim(0, 120)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Saturation distribution
ax = axes[0, 1]
ax.hist(h_sat, bins=30, color='#27ae60', alpha=0.75,
        label=f'Healthy   mean={h_sat.mean():.1f}', density=True)
ax.hist(u_sat, bins=30, color='#e67e22', alpha=0.75,
        label=f'Unhealthy mean={u_sat.mean():.1f}', density=True)
ax.set_title('Saturation Distribution')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Color Health Score distribution
ax = axes[1, 0]
ax.hist(h_scores, bins=20, color='#27ae60', alpha=0.75,
        label=f'Healthy   mean={h_scores.mean():.1f}', density=True)
ax.hist(u_scores, bins=20, color='#e67e22', alpha=0.75,
        label=f'Unhealthy mean={u_scores.mean():.1f}', density=True)
ax.axvline(70, color='green', linestyle='--', alpha=0.7, label='Healthy threshold (70)')
ax.axvline(30, color='red',   linestyle='--', alpha=0.7, label='Unhealthy threshold (30)')
ax.set_title('COLOR HEALTH SCORE (0-100)', fontweight='bold')
ax.set_xlabel('Score (100 = fully green/healthy)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Color Health Score scale visualization
ax = axes[1, 1]
import matplotlib.colors as mcolors
gradient = np.linspace(0, 1, 300).reshape(1, -1)
cmap = mcolors.LinearSegmentedColormap.from_list(
    'health', ['#8B4513', '#e67e22', '#f1c40f', '#a8d96c', '#27ae60'])
ax.imshow(gradient, aspect='auto', cmap=cmap, extent=[0, 100, 0, 1])
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.axvline(30, color='black', linewidth=2, linestyle='--')
ax.axvline(70, color='black', linewidth=2, linestyle='--')
ax.text(15, 0.5, 'UNHEALTHY\n(0-29)', ha='center', va='center',
        fontweight='bold', fontsize=9, color='white')
ax.text(50, 0.5, 'MODERATE\n(30-69)', ha='center', va='center',
        fontweight='bold', fontsize=9, color='black')
ax.text(85, 0.5, 'HEALTHY\n(70-100)', ha='center', va='center',
        fontweight='bold', fontsize=9, color='white')
ax.scatter([h_scores.mean()], [0.85], color='darkgreen', s=120, zorder=5,
           label=f'Healthy avg: {h_scores.mean():.0f}')
ax.scatter([u_scores.mean()], [0.15], color='darkred',  s=120, zorder=5,
           label=f'Unhealthy avg: {u_scores.mean():.0f}')
ax.set_title('Color Health Score Scale', fontweight='bold')
ax.set_xlabel('Health Score (0 = Brown/Yellow   →   100 = Pure Green)')
ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'color_health_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Score separation: {score_sep:.1f} pts")
print(f"Healthy avg score:   {h_scores.mean():.1f} / 100")
print(f"Unhealthy avg score: {u_scores.mean():.1f} / 100")

## 6. Sample Images Visualization

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Sample Training Images — Healthy (Green) vs Unhealthy (Yellowing/Browning)',
             fontsize=13, fontweight='bold')

cls_colors = {'healthy': '#27ae60', 'unhealthy': '#e67e22'}
for row, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(DATASET_DIR, 'train', cls)
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith(VALID_EXT)]
    chosen = random.sample(files, min(6, len(files)))
    for col, fname in enumerate(chosen):
        fp  = os.path.join(cls_dir, fname)
        img = keras_image.load_img(fp, target_size=(IMG_SIZE, IMG_SIZE))
        sc  = compute_color_health_score(fp)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        axes[row, col].set_title(f'Score: {sc:.0f}/100', fontsize=8,
                                  color=cls_colors[cls], fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(cls.upper(), color=cls_colors[cls],
                                       fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Data Generators

### CRITICAL: No Color Augmentation
The model learns **green = healthy / yellow-brown = unhealthy** from color.  
Any color augmentation (brightness, hue, channel shift) corrupts the labels.

Only **geometric augmentation** is applied (rotation, flip, zoom, shift).

In [ ]:
# Geometric-only augmentation — color is untouched
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.20,
    height_shift_range=0.20,
    horizontal_flip=True,
    vertical_flip=False,        # leaves have an orientation
    zoom_range=0.20,
    shear_range=0.10,
    fill_mode='nearest'
    # NO brightness_range   <- would make healthy look unhealthy
    # NO channel_shift_range <- would corrupt green vs yellow signal
)

eval_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=True,
    seed=42
)
val_gen = eval_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)
test_gen = eval_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print(f"Train : {train_gen.samples}")
print(f"Val   : {val_gen.samples}")
print(f"Test  : {test_gen.samples}")
print(f"Class indices: {train_gen.class_indices}")
print(f"Test >= 500   : {'YES' if test_gen.samples >= 500 else 'NO'}")

## 8. Class Weights

In [ ]:
train_labels = train_gen.classes
cw = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict = {i: w for i, w in enumerate(cw)}

print("Class Weights (balanced):")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  {i} ({cls:<12}): {class_weight_dict[i]:.4f}")

imbalance = max(len(h_train), len(u_train)) / min(len(h_train), len(u_train))
print(f"\nClass imbalance ratio: {imbalance:.2f}x")

## 9. Focal Loss + Label Smoothing

**Label Smoothing (0.1)** prevents overconfidence:  
- Without it: model outputs 99–100% → unrealistic  
- With it: outputs capped at ~95% → realistic confidence scores

In [ ]:
def focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1):
    """
    Focal Loss with Label Smoothing.
    - gamma=2.0        : focus on harder examples
    - alpha=0.25       : class balancing weight
    - label_smoothing  : prevents 100% overconfident outputs
    """
    n_classes = 2

    def loss_fn(y_true, y_pred):
        y_smooth = y_true * (1.0 - label_smoothing) + (label_smoothing / n_classes)
        eps = tf.keras.backend.epsilon()
        y_pred_c = tf.keras.backend.clip(y_pred, eps, 1.0 - eps)
        ce = -y_smooth * tf.keras.backend.log(y_pred_c)
        fw = tf.keras.backend.pow(1.0 - y_pred_c, gamma)
        return tf.keras.backend.sum(alpha * fw * ce, axis=-1)

    return loss_fn

print("Focal Loss + Label Smoothing ready")
print("  gamma=2.0           -> harder examples get more focus")
print("  alpha=0.25          -> class balancing")
print("  label_smoothing=0.1 -> no overconfident 100% outputs")

## 10. Build Model — EfficientNetB0

**Why EfficientNetB0 for color-based classification?**
- ImageNet pretrained → already learned green/yellow/brown color features
- Compound scaling → better texture + color feature extraction vs MobileNetV2
- Stronger baseline for subtle color differences

In [ ]:
def build_model():
    """
    EfficientNetB0 feature extractor + custom classification head.
    Generator outputs [0,1] -> Rescaling layer maps back to [0,255]
    as required by EfficientNetB0's built-in preprocessing.
    """
    base = EfficientNetB0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # Phase 1: frozen

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(scale=255.0)(inputs)   # [0,1] -> [0,255]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)

    return keras.Model(inputs, outputs), base


model, base_model = build_model()
print(f"Architecture  : EfficientNetB0 + custom head")
print(f"Base layers   : {len(base_model.layers)}")
print(f"Total params  : {model.count_params():,}")
model.summary()

## 11. Phase 1 — Frozen Base Training

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR_PHASE1),
    loss=focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p1 = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

print("=" * 65)
print("PHASE 1 — Frozen EfficientNetB0 (custom head only)")
print("=" * 65)

t0 = time.time()
history_p1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks_p1,
    verbose=1
)
p1_min = (time.time() - t0) / 60
best_p1 = max(history_p1.history['val_accuracy'])
print(f"\nPhase 1: {p1_min:.1f} min | Best val acc: {best_p1*100:.2f}%")

## 12. Phase 2 — Fine-tuning

In [ ]:
# Unfreeze the last 50 layers of EfficientNetB0
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(LR_PHASE2),
    loss=focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p2 = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1),
]

print(f"Fine-tuning from layer {fine_tune_at} / {len(base_model.layers)}")
print(f"Trainable layers: {sum(1 for l in model.layers if l.trainable)}")
print("=" * 65)
print("PHASE 2 — Fine-tuning color-sensitive layers")
print("=" * 65)

t0 = time.time()
history_p2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks_p2,
    verbose=1
)
p2_min = (time.time() - t0) / 60
total_min = p1_min + p2_min
best_p2 = max(history_p2.history['val_accuracy'])
print(f"\nPhase 2: {p2_min:.1f} min | Total: {total_min:.1f} min | Best val acc: {best_p2*100:.2f}%")

## 13. Training History

In [ ]:
hist = {
    k: history_p1.history[k] + history_p2.history[k]
    for k in ['accuracy', 'val_accuracy', 'loss', 'val_loss']
}
p1_end = len(history_p1.history['accuracy']) - 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History — Leaf Health v3 (EfficientNetB0)',
             fontsize=13, fontweight='bold')

for ax, (metric, title) in zip(axes,
                                [('accuracy', 'Accuracy'), ('loss', 'Loss')]):
    ax.plot(hist[metric],          label='Train', linewidth=2)
    ax.plot(hist[f'val_{metric}'], label='Val',   linewidth=2)
    ax.axvline(p1_end, color='red', linestyle='--', alpha=0.7,
               label='Fine-tune start')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

final_train_acc = hist['accuracy'][-1]
final_val_acc   = hist['val_accuracy'][-1]
gap = abs(final_train_acc - final_val_acc)
print(f"Train: {final_train_acc*100:.2f}%  Val: {final_val_acc*100:.2f}%  Gap: {gap*100:.2f}%")
print(f"Overfitting: {'OK' if gap < 0.05 else 'Minor' if gap < 0.10 else 'HIGH'}")

## 14. Evaluate on Test Set

In [ ]:
best_model = keras.models.load_model(
    os.path.join(MODEL_DIR, 'best_model.keras'),
    custom_objects={
        'loss_fn': focal_loss_smoothed(gamma=2.0, alpha=0.25, label_smoothing=0.1)
    }
)

test_gen.reset()
preds    = best_model.predict(test_gen, verbose=1)
y_true   = test_gen.classes
y_pred   = np.argmax(preds, axis=1)
y_conf   = np.max(preds, axis=1)
test_acc = np.mean(y_true == y_pred)

print(f"\nTest samples  : {len(y_true)}")
print(f"Test accuracy : {test_acc*100:.2f}%  {'(>=95% target)' if test_acc >= 0.95 else '(below 95% target)'}")
print(f"Max confidence: {y_conf.max()*100:.2f}%")
print(f"Mean confidence: {y_conf.mean()*100:.2f}%")

## 15. Class-wise Metrics — Supervisor Requirements

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None
)
macro_p = np.mean(precision)
macro_r = np.mean(recall)
macro_f = np.mean(f1)

print("=" * 90)
print("CLASS-WISE METRICS — leaf_health_v3")
print("=" * 90)
print(f"\n{'Class':<15} {'Precision':>12} {'Recall':>12} {'F1-Score':>12} {'Support':>10}")
print("-" * 65)
for i, cls in enumerate(CLASS_NAMES):
    print(f"{cls:<15} {precision[i]*100:>11.2f}% {recall[i]*100:>11.2f}%"
          f" {f1[i]*100:>11.2f}% {support[i]:>10}")
print("-" * 65)
print(f"{'Macro Avg':<15} {macro_p*100:>11.2f}% {macro_r*100:>11.2f}%"
      f" {macro_f*100:>11.2f}%")
print("=" * 90)

# --- Supervisor Requirement Checks ---
print("\n" + "=" * 90)
print("SUPERVISOR REQUIREMENT CHECKS")
print("=" * 90)

# Check 1: P/R/F1 close within each class (max diff < 10%)
print("\n[1] P, R, F1 close to each other per class (max diff < 10%):")
p1_ok_all = True
for i, cls in enumerate(CLASS_NAMES):
    p, r, f = precision[i], recall[i], f1[i]
    max_d = max(abs(p - r), abs(p - f), abs(r - f))
    ok = max_d < 0.10
    if not ok:
        p1_ok_all = False
    print(f"  {cls.upper():12} P={p*100:.2f}%  R={r*100:.2f}%  F1={f*100:.2f}%"
          f"  max_diff={max_d*100:.2f}%  {'OK' if ok else 'NEEDS IMPROVEMENT'}")

# Check 2: Similar across classes
f1_diff = abs(f1[0] - f1[1])
p_diff  = abs(precision[0] - precision[1])
r_diff  = abs(recall[0] - recall[1])
print(f"\n[2] Similar values across both classes (diff < 10%):")
print(f"  F1 diff        : {f1_diff*100:.2f}%  {'OK' if f1_diff < 0.10 else 'NEEDS IMPROVEMENT'}")
print(f"  Precision diff : {p_diff*100:.2f}%  {'OK' if p_diff < 0.10 else 'NEEDS IMPROVEMENT'}")
print(f"  Recall diff    : {r_diff*100:.2f}%  {'OK' if r_diff < 0.10 else 'NEEDS IMPROVEMENT'}")

# Check 3: Test size
print(f"\n[3] Test set size: {len(y_true)}  {'OK (>=500)' if len(y_true) >= 500 else 'NOT ENOUGH'}")

# Check 4: Accuracy target
print(f"\n[4] Accuracy: {test_acc*100:.2f}%  {'OK (>=95%)' if test_acc >= 0.95 else 'below 95% target'}")
print("=" * 90)

## 16. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confusion Matrix — Leaf Health v3', fontsize=13, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1],
            cbar_kws={'label': '%'})
axes[1].set_title('Percentages (%)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

for i, tc in enumerate(CLASS_NAMES):
    for j, pc in enumerate(CLASS_NAMES):
        sym = 'CORRECT' if i == j else 'WRONG  '
        print(f"  {sym} | True: {tc:<12} Pred: {pc:<12}"
              f" {cm[i,j]:>5} ({cm_pct[i,j]:.1f}%)")

## 17. Full Classification Report

In [ ]:
print("=" * 70)
print("FULL CLASSIFICATION REPORT — leaf_health_v3")
print("=" * 70)
print(classification_report(y_true, y_pred,
                             target_names=CLASS_NAMES, digits=4))

# Confidence distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Confidence Distribution — v3 (Label Smoothing Applied)',
             fontsize=12, fontweight='bold')

correct_c = y_conf[y_true == y_pred]
wrong_c   = y_conf[y_true != y_pred]

axes[0].hist(correct_c, bins=25, color='#27ae60', edgecolor='black', alpha=0.8)
axes[0].axvline(correct_c.mean(), color='darkgreen', linestyle='--',
                label=f'Mean: {correct_c.mean():.3f}')
axes[0].set_title(f'Correct (n={len(correct_c)})')
axes[0].legend()

if len(wrong_c):
    axes[1].hist(wrong_c, bins=25, color='#e74c3c', edgecolor='black', alpha=0.8)
    axes[1].axvline(wrong_c.mean(), color='darkred', linestyle='--',
                    label=f'Mean: {wrong_c.mean():.3f}')
    axes[1].set_title(f'Wrong (n={len(wrong_c)})')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'No wrong predictions!',
                 ha='center', va='center', fontsize=14, color='green',
                 transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confidence_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Max confidence : {y_conf.max()*100:.2f}%")
print(f"Mean confidence: {y_conf.mean()*100:.2f}%")

## 18. Color Health Score Validation

Compute the Color Health Score (0–100) for all test images and confirm the score separates healthy from unhealthy leaves.

In [ ]:
print("Computing Color Health Scores for all test images...")
filenames = test_gen.filenames
color_scores = []
for fname in filenames:
    fp = os.path.join(DATASET_DIR, 'test', fname)
    color_scores.append(compute_color_health_score(fp))
color_scores = np.array(color_scores)

h_scores_test = color_scores[y_true == 0]   # index 0 = healthy
u_scores_test = color_scores[y_true == 1]   # index 1 = unhealthy

print(f"\nColor Health Score Statistics (Test Set):")
print(f"  {'Class':<12} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print(f"  {'-'*50}")
print(f"  {'Healthy':12} {h_scores_test.mean():>8.1f} {h_scores_test.std():>8.1f}"
      f" {h_scores_test.min():>8.1f} {h_scores_test.max():>8.1f}")
print(f"  {'Unhealthy':12} {u_scores_test.mean():>8.1f} {u_scores_test.std():>8.1f}"
      f" {u_scores_test.min():>8.1f} {u_scores_test.max():>8.1f}")

# Score-based prediction accuracy (simple threshold)
score_threshold = (h_scores_test.mean() + u_scores_test.mean()) / 2
score_pred = (color_scores < score_threshold).astype(int)   # <threshold = unhealthy (1)
score_acc = np.mean(score_pred == y_true)
print(f"\n  Optimal threshold        : {score_threshold:.1f}")
print(f"  Score-only accuracy      : {score_acc*100:.2f}%")
print(f"  DL model accuracy        : {test_acc*100:.2f}%")
print(f"  DL model is {'BETTER' if test_acc > score_acc else 'SIMILAR'} than color-only approach")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Color Health Score Validation — Test Set', fontsize=13, fontweight='bold')

# Score distribution by class
ax = axes[0]
ax.hist(h_scores_test, bins=25, color='#27ae60', alpha=0.75,
        label=f'Healthy   mean={h_scores_test.mean():.1f}', density=True)
ax.hist(u_scores_test, bins=25, color='#e67e22', alpha=0.75,
        label=f'Unhealthy mean={u_scores_test.mean():.1f}', density=True)
ax.axvline(score_threshold, color='black', linestyle='--',
           label=f'Threshold: {score_threshold:.1f}')
ax.set_title('Score Distribution by True Class')
ax.set_xlabel('Color Health Score (0-100)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Score vs DL confidence scatter
ax = axes[1]
for cls_idx, (cls, col) in enumerate([('healthy', '#27ae60'), ('unhealthy', '#e67e22')]):
    mask = y_true == cls_idx
    ax.scatter(color_scores[mask], y_conf[mask], c=col, alpha=0.3, s=10, label=cls)
ax.set_xlabel('Color Health Score (0-100)')
ax.set_ylabel('DL Model Confidence')
ax.set_title('Color Score vs DL Confidence')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Score scale bar with avg markers
ax = axes[2]
import matplotlib.colors as mcolors
gradient = np.linspace(0, 1, 300).reshape(1, -1)
cmap = mcolors.LinearSegmentedColormap.from_list(
    'health', ['#8B4513', '#e67e22', '#f1c40f', '#a8d96c', '#27ae60'])
ax.imshow(gradient, aspect='auto', cmap=cmap, extent=[0, 100, 0, 1])
ax.set_xlim(0, 100); ax.set_yticks([])
ax.axvline(30, color='black', linewidth=2, linestyle='--', alpha=0.7)
ax.axvline(70, color='black', linewidth=2, linestyle='--', alpha=0.7)
ax.scatter([h_scores_test.mean()], [0.8], color='darkgreen', s=150, zorder=5,
           label=f'Healthy avg: {h_scores_test.mean():.1f}')
ax.scatter([u_scores_test.mean()], [0.2], color='darkred',  s=150, zorder=5,
           label=f'Unhealthy avg: {u_scores_test.mean():.1f}')
ax.text(15, 0.5, 'UNHEALTHY', ha='center', va='center',
        fontweight='bold', fontsize=10, color='white')
ax.text(50, 0.5, 'MODERATE',  ha='center', va='center',
        fontweight='bold', fontsize=10, color='black')
ax.text(85, 0.5, 'HEALTHY',   ha='center', va='center',
        fontweight='bold', fontsize=10, color='white')
ax.set_title('Color Health Score Scale', fontweight='bold')
ax.set_xlabel('Score (0 = Yellow/Brown   ->   100 = Pure Green)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'color_score_validation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 19. Sample Predictions with Color Health Score

In [ ]:
correct_idx = [i for i in range(len(y_true)) if y_true[i] == y_pred[i]]
wrong_idx   = [i for i in range(len(y_true)) if y_true[i] != y_pred[i]]
print(f"Total  : {len(y_true)}")
print(f"Correct: {len(correct_idx)}")
print(f"Wrong  : {len(wrong_idx)}")


def show_prediction_grid(indices, title, border_color, n=10):
    n = min(n, len(indices))
    if n == 0:
        print(f"{title}: nothing to show")
        return
    cols = 5
    rows = max(1, (n + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3.8 * rows))
    fig.suptitle(title, fontsize=13, fontweight='bold', color=border_color)
    if rows == 1:
        axes = axes.reshape(1, -1)

    chosen = random.sample(indices, n)
    for idx, i in enumerate(chosen):
        r, c = idx // cols, idx % cols
        fp   = os.path.join(DATASET_DIR, 'test', filenames[i])
        img  = keras_image.load_img(fp, target_size=(IMG_SIZE, IMG_SIZE))
        sc   = color_scores[i]
        score_label = ('Healthy' if sc >= 70
                       else 'Moderate' if sc >= 30 else 'Unhealthy')
        axes[r, c].imshow(img)
        axes[r, c].axis('off')
        axes[r, c].set_title(
            f"True: {CLASS_NAMES[y_true[i]]}\n"
            f"Pred: {CLASS_NAMES[y_pred[i]]} ({y_conf[i]*100:.1f}%)\n"
            f"Color Score: {sc:.0f}/100 ({score_label})",
            fontsize=7, color=border_color
        )

    for idx in range(n, rows * cols):
        axes[idx // cols, idx % cols].axis('off')

    plt.tight_layout()
    fname = ('correct_predictions.png'
             if 'Correct' in title else 'wrong_predictions.png')
    plt.savefig(os.path.join(MODEL_DIR, fname), dpi=150, bbox_inches='tight')
    plt.show()


show_prediction_grid(correct_idx, 'CORRECT Predictions', 'green')
show_prediction_grid(wrong_idx,
                     f'WRONG Predictions ({len(wrong_idx)} total)', 'red')

## 20. Inference Helper — predict_leaf_health()

A ready-to-use function for the Flask API integration.

In [ ]:
def predict_leaf_health(img_path, model_obj=None):
    """
    Predict leaf health for a single image.

    Returns dict:
      {
        'prediction'         : 'healthy' | 'unhealthy',
        'confidence'         : float (0-1),
        'confidence_pct'     : float (0-100),
        'color_health_score' : float (0-100),
        'color_label'        : 'Healthy' | 'Moderate' | 'Unhealthy',
        'probabilities'      : {'healthy': float, 'unhealthy': float},
      }
    """
    m = model_obj if model_obj is not None else best_model

    # DL prediction
    img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras_image.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, 0)
    probs = m.predict(arr, verbose=0)[0]

    pred_idx  = int(np.argmax(probs))
    pred_cls  = CLASS_NAMES[pred_idx]
    conf      = float(probs[pred_idx])

    # Color Health Score
    score = compute_color_health_score(img_path)
    label = ('Healthy' if score >= 70 else
             'Moderate' if score >= 30 else 'Unhealthy')

    return {
        'prediction'         : pred_cls,
        'confidence'         : conf,
        'confidence_pct'     : round(conf * 100, 2),
        'color_health_score' : score,
        'color_label'        : label,
        'probabilities'      : {
            cls: round(float(probs[i]) * 100, 2)
            for i, cls in enumerate(CLASS_NAMES)
        },
    }


# --- Quick test on 5 random test images ---
print("Quick inference test (5 random test images):")
print("-" * 80)
test_files = random.sample(filenames, 5)
for fname in test_files:
    fp     = os.path.join(DATASET_DIR, 'test', fname)
    result = predict_leaf_health(fp)
    true_label = fname.split(os.sep)[0]
    correct = 'OK' if result['prediction'] == true_label else 'WRONG'
    print(f"  [{correct}] True={true_label:<10} "
          f"Pred={result['prediction']:<10} "
          f"Conf={result['confidence_pct']:.1f}% "
          f"Score={result['color_health_score']:.0f}/100 ({result['color_label']})")

## 21. Save Model Info

In [ ]:
model_info = {
    'model_name'   : 'leaf_health_v3',
    'architecture' : 'EfficientNetB0',
    'version'      : 'v3',
    'classes'      : CLASS_NAMES,
    'num_classes'  : 2,
    'input_shape'  : [IMG_SIZE, IMG_SIZE, 3],
    'new_in_v3': [
        'Color Health Score (0-100): green pixel ratio via HSV analysis',
        'Score labels: Healthy (70-100) | Moderate (30-69) | Unhealthy (0-29)',
        'LR Phase 2 reduced to 3e-5 for gentler fine-tuning',
        'EarlyStopping patience increased to 8 epochs',
        'predict_leaf_health() helper included for API integration'
    ],
    'key_design_decisions': [
        'NO color augmentation - preserves green vs yellow/brown color signal',
        'Geometric augmentation only (rotation/flip/zoom/shift)',
        'EfficientNetB0 - better color feature extraction than MobileNetV2',
        'Label smoothing 0.1 - prevents overconfident 100% outputs',
        'Data re-split 70/15/15 - ensures 500+ test images'
    ],
    'color_health_score': {
        'description'   : 'Green pixel ratio (HSV) mapped to 0-100 scale',
        'formula'       : 'score = (green_pixels / total_saturated_pixels) * 100',
        'green_hue_range': '35-85 (OpenCV scale 0-180)',
        'sat_threshold' : 40,
        'labels': {
            '70-100': 'Healthy',
            '30-69' : 'Moderate',
            '0-29'  : 'Unhealthy'
        }
    },
    'training': {
        'phase1_epochs'          : PHASE1_EPOCHS,
        'phase2_epochs'          : PHASE2_EPOCHS,
        'batch_size'             : BATCH_SIZE,
        'lr_phase1'              : LR_PHASE1,
        'lr_phase2'              : LR_PHASE2,
        'loss_function'          : 'Focal Loss (gamma=2.0, alpha=0.25) + Label Smoothing 0.1',
        'optimizer'              : 'Adam',
        'augmentation'           : 'Geometric only — NO color changes',
        'class_weights'          : {str(k): float(v) for k, v in class_weight_dict.items()},
        'training_time_minutes'  : round(total_min, 1),
        'final_train_accuracy'   : float(final_train_acc),
        'final_val_accuracy'     : float(final_val_acc)
    },
    'data': {
        'healthy_source'   : 'healthy-leaves/',
        'unhealthy_source' : 'unhealthy-yellowing/',
        'split'            : '70% train / 15% val / 15% test',
        'train_samples'    : train_gen.samples,
        'val_samples'      : val_gen.samples,
        'test_samples'     : test_gen.samples,
        'train_healthy'    : data_summary['train']['healthy'],
        'train_unhealthy'  : data_summary['train']['unhealthy']
    },
    'test_performance': {
        'accuracy'            : float(test_acc),
        'macro_precision'     : float(macro_p),
        'macro_recall'        : float(macro_r),
        'macro_f1'            : float(macro_f),
        'healthy_precision'   : float(precision[0]),
        'healthy_recall'      : float(recall[0]),
        'healthy_f1'          : float(f1[0]),
        'unhealthy_precision' : float(precision[1]),
        'unhealthy_recall'    : float(recall[1]),
        'unhealthy_f1'        : float(f1[1])
    },
    'color_score_stats': {
        'healthy_mean_score'   : float(h_scores_test.mean()),
        'unhealthy_mean_score' : float(u_scores_test.mean()),
        'optimal_threshold'    : float(score_threshold),
        'score_only_accuracy'  : float(score_acc)
    },
    'supervisor_checks': {
        'test_samples_500_plus'   : bool(test_gen.samples >= 500),
        'accuracy_95_plus'        : bool(test_acc >= 0.95),
        'healthy_prf_balanced'    : bool(
            max(abs(precision[0]-recall[0]),
                abs(precision[0]-f1[0]),
                abs(recall[0]-f1[0])) < 0.10),
        'unhealthy_prf_balanced'  : bool(
            max(abs(precision[1]-recall[1]),
                abs(precision[1]-f1[1]),
                abs(recall[1]-f1[1])) < 0.10),
        'cross_class_balanced'    : bool(abs(f1[0]-f1[1]) < 0.10),
        'no_overconfidence'       : bool(float(y_conf.max()) < 0.999)
    }
}

info_path = os.path.join(MODEL_DIR, 'model_info.json')
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Saved: {info_path}")
print("\nFiles in model directory:")
for fn in sorted(os.listdir(MODEL_DIR)):
    kb = os.path.getsize(os.path.join(MODEL_DIR, fn)) / 1024
    print(f"  {fn:<45} {kb:>8.1f} KB")

## 22. Final Summary

In [ ]:
chk = model_info['supervisor_checks']

print("\n" + "=" * 85)
print("  COCONUT LEAF HEALTH MODEL v3 — FINAL SUMMARY")
print("=" * 85)
print()
print("  Dataset:")
print(f"    healthy-leaves/  +  unhealthy-yellowing/")
print(f"    Split 70/15/15 -> Train:{train_gen.samples} | Val:{val_gen.samples} | Test:{test_gen.samples}")
print()
print("  Model:")
print(f"    Architecture  : EfficientNetB0 (ImageNet pretrained)")
print(f"    Loss          : Focal Loss + Label Smoothing 0.1")
print(f"    Augmentation  : Geometric only (NO color changes)")
print(f"    Training time : {total_min:.1f} minutes")
print()
print("  Test Performance:")
print(f"    Accuracy        : {test_acc*100:.2f}%  {'OK' if test_acc >= 0.95 else 'BELOW TARGET'}")
print(f"    Macro Precision : {macro_p*100:.2f}%")
print(f"    Macro Recall    : {macro_r*100:.2f}%")
print(f"    Macro F1-Score  : {macro_f*100:.2f}%")
print()
print("  Class-wise Metrics:")
for i, cls in enumerate(CLASS_NAMES):
    mx = max(abs(precision[i]-recall[i]),
             abs(precision[i]-f1[i]),
             abs(recall[i]-f1[i]))
    print(f"    {cls.upper():12}  P={precision[i]*100:.2f}%  R={recall[i]*100:.2f}%"
          f"  F1={f1[i]*100:.2f}%  {'OK' if mx < 0.10 else 'UNBALANCED'}")
print()
print("  Color Health Score (v3 New Feature):")
print(f"    Healthy avg score   : {h_scores_test.mean():.1f} / 100")
print(f"    Unhealthy avg score : {u_scores_test.mean():.1f} / 100")
print(f"    Separation          : {abs(h_scores_test.mean()-u_scores_test.mean()):.1f} pts")
print(f"    Threshold (auto)    : {score_threshold:.1f}")
print(f"    Scale: 0-29=Unhealthy | 30-69=Moderate | 70-100=Healthy")
print()
print("  Supervisor Requirement Checks:")
print(f"    [{'OK' if chk['test_samples_500_plus']  else 'FAIL'}] Test set >= 500         : {test_gen.samples}")
print(f"    [{'OK' if chk['accuracy_95_plus']       else 'WARN'}] Accuracy >= 95%         : {test_acc*100:.2f}%")
print(f"    [{'OK' if chk['healthy_prf_balanced']   else 'WARN'}] Healthy  P/R/F1 balanced")
print(f"    [{'OK' if chk['unhealthy_prf_balanced'] else 'WARN'}] Unhealthy P/R/F1 balanced")
print(f"    [{'OK' if chk['cross_class_balanced']   else 'WARN'}] Cross-class balanced")
print(f"    [{'OK' if chk['no_overconfidence']      else 'WARN'}] No overconfidence (max={y_conf.max()*100:.1f}%)")
print()
print(f"  Model saved: {MODEL_DIR}/best_model.keras")
print(f"  Info saved : {MODEL_DIR}/model_info.json")
print()
print("=" * 85)
print("              TRAINING COMPLETE — leaf_health_v3")
print("=" * 85)